In [ ]:
import pandas as pd
import pytorch_lightning as pl
from torch.utils.data.dataloader import DataLoader
from torchvision.transforms import Compose, Normalize, Resize, ToTensor

from dataset.slope_dataset import SlopeDataset
from lib.utils.path import output_path
from models.wheel_safe_classifier import WheelSafeClassifier

# 이미지 크기는 모델에 맞게 조절 (EfficientNet 기본 224, 240 등)
data_transform = Compose(
    [
        Resize((224, 224)),
        ToTensor(),
        Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ]
)

# CSV 또는 DataFrame 로드 (올려주신 이미지 기준 예시)
df = pd.read_csv(output_path() / 'slope_labels_007.csv')

train_dataset = SlopeDataset(df, transform=data_transform)
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)

# --- 모델 생성 및 학습 ---
model_system = WheelSafeClassifier(model_name='efficientnet_b0')

trainer = pl.Trainer(
    max_epochs=50,
    accelerator='gpu',
    devices=1,
    precision=16,  # 경사도 연산 가속
)

trainer.fit(model_system, train_loader)